In [3]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(-1)
X_val   = X_val.unsqueeze(-1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,      # feature per timestep
            hidden_size=64,    # memory size
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        out, _ = self.lstm(x)     # out: (batch, seq_len, hidden)

        out = out[:, -1, :]       # take last timestep output

        out = self.fc(out)
        return out

model = LSTMModel()

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy()))

Epoch 1, Train Loss: 19.5342
Epoch 2, Train Loss: 5.0726
Epoch 3, Train Loss: 4.4628
Epoch 4, Train Loss: 4.1817
Epoch 5, Train Loss: 3.1695
Epoch 6, Train Loss: 4.3378
Epoch 7, Train Loss: 3.9224
Epoch 8, Train Loss: 3.0831
Epoch 9, Train Loss: 2.4715
Epoch 10, Train Loss: 2.2858
Epoch 11, Train Loss: 2.0003
Epoch 12, Train Loss: 2.0521
Epoch 13, Train Loss: 1.5882
Epoch 14, Train Loss: 1.2311
Epoch 15, Train Loss: 1.4527
Epoch 16, Train Loss: 0.9815
Epoch 17, Train Loss: 0.8617
Epoch 18, Train Loss: 0.7178
Epoch 19, Train Loss: 0.7470
Epoch 20, Train Loss: 0.6344
Total evaluated samples: 1018
Validation Accuracy: 0.8821218013763428
[[898 113]
 [  7   0]]
              precision    recall  f1-score   support

           0       0.99      0.89      0.94      1011
           1       0.00      0.00      0.00         7

    accuracy                           0.88      1018
   macro avg       0.50      0.44      0.47      1018
weighted avg       0.99      0.88      0.93      1018



In [5]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

data = pd.read_csv('exoTrain.csv')

X = data.drop('LABEL', axis=1).values
y = data['LABEL'].values - 1  # make labels 0,1

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# convert AFTER split
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)

# reshape for LSTM
X_train = X_train.unsqueeze(-1)
X_val   = X_val.unsqueeze(-1)


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,      # feature per timestep
            hidden_size=64,    # memory size
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 2, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        out, _ = self.lstm(x)     # out: (batch, seq_len, hidden)

        out = out[:, -1, :]       # take last timestep output

        out = self.fc(out)
        return out

model = LSTMModel()

class_counts = torch.bincount(y_train)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[y_train]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        outputs = model(xb)
        loss = criterion(outputs, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        outputs = model(xb)
        _, preds = torch.max(outputs, 1)

        all_preds.append(preds)
        all_labels.append(yb)

# combine batches
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("Total evaluated samples:", len(all_preds))

accuracy = (all_preds == all_labels).float().mean()
print("Validation Accuracy:", accuracy.item())

print(confusion_matrix(all_labels.numpy(), all_preds.numpy()))
print(classification_report(all_labels.numpy(), all_preds.numpy())) ## this is bidirectional search 

Epoch 1, Train Loss: 21.7124
Epoch 2, Train Loss: 5.2885
Epoch 3, Train Loss: 4.2604
Epoch 4, Train Loss: 3.8218
Epoch 5, Train Loss: 2.5629
Epoch 6, Train Loss: 1.9370
Epoch 7, Train Loss: 2.8353
Epoch 8, Train Loss: 1.9634
Epoch 9, Train Loss: 4.2721
Epoch 10, Train Loss: 2.3142
Epoch 11, Train Loss: 1.7559
Epoch 12, Train Loss: 1.5722
Epoch 13, Train Loss: 1.5832
Epoch 14, Train Loss: 1.9543
Epoch 15, Train Loss: 1.3983
Epoch 16, Train Loss: 1.0937
Epoch 17, Train Loss: 1.0744
Epoch 18, Train Loss: 0.9781
Epoch 19, Train Loss: 0.9139
Epoch 20, Train Loss: 0.8453
Total evaluated samples: 1018
Validation Accuracy: 0.7318271398544312
[[744 267]
 [  6   1]]
              precision    recall  f1-score   support

           0       0.99      0.74      0.84      1011
           1       0.00      0.14      0.01         7

    accuracy                           0.73      1018
   macro avg       0.50      0.44      0.43      1018
weighted avg       0.99      0.73      0.84      1018

